In [61]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict,Annotated
import os
import operator
import math
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal

load_dotenv()

True

In [62]:

endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )
model=ChatHuggingFace(llm=endpoint)

In [63]:
class SentimentSchema(BaseModel):
    sentiment:Literal["positive","negative","neutral"]=Field(description="The sentiment of the review")

In [64]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [65]:
structured_model=model.with_structured_output(SentimentSchema,method="json_schema")
structured_model2=model.with_structured_output(DiagnosisSchema,method="json_schema")

In [66]:
prompt='What is the sentiment of the following review? Return JSON with key "sentiment" as one of: positive, negative, neutral. - The software is great!'
result=structured_model.invoke(prompt)
print(result)


{'sentiment': 'positive'}


In [67]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive","negative","neutral"]
    diagnosis:str
    response:str


In [68]:
def find_sentiment(state:ReviewState)->ReviewState:
    prompt=f'What is the sentiment of the following review? Return JSON with key "sentiment" as one of: positive, negative, neutral. - {state["review"]}'
    result=structured_model.invoke(prompt)["sentiment"]
    return {"sentiment":result}


In [69]:
def check_sentiment(state:ReviewState) -> Literal['positive_response',"run_diagnosis"]:
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [70]:
from pydantic_core.core_schema import model_field


def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

    # json to dictionary through model_dump()

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}
    

In [71]:
graph=StateGraph(ReviewState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("positive_response",positive_response)
graph.add_node("negative_response",negative_response)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge("positive_response",END)
graph.add_edge("run_diagnosis","negative_response")

workflow=graph.compile()

initial_state={"review":"The software is great!"}
workflow.invoke(initial_state)


{'review': 'The software is great!',
 'sentiment': 'positive',
 'response': 'Here\'s a warm thank-you message in response to the review:\n\n"Thank you so much for your kind words about our software! We\'re thrilled to hear that you\'re enjoying it. Your feedback means a lot to us, and we\'re glad you\'re finding it helpful.\n\nIf you have a moment, we\'d love to hear more about your experience with our software. Please take a minute to leave a review on our website - it really helps us understand how we can continue to improve and provide the best possible experience for our users. Your feedback is invaluable to us! Thanks again for your support!"'}